# Bronze - ecommerce_rastreamento_entregas

Desenvolvido por: Ygor Moraes

Este notebook lê o arquivo `ecommerce_rastreamento_entregas.csv` da camada Raw e grava os dados na Bronze em Delta.

Regras aplicadas:
- preservar os dados brutos como string;
- adicionar auditoria com `bronze_ingested_at` e `bronze_source_file`;
- gerar `bronze_record_hash` para controle incremental;
- particionar por `ano` e `mes` a partir de `dt_evento`;
- gravar em modo append, evitando reprocessar registros já carregados.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Bronze de rastreamento.

from pyspark.sql.functions import (
    col,
    count,
    when,
    current_timestamp,
    to_timestamp,
    year,
    month,
    sha2,
    concat_ws,
    coalesce,
    lit
)

SOURCE_FILE = "ecommerce_rastreamento_entregas.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

BRONZE_TABLE = "ecommerce_rastreamento_entregas"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao"
]

PARTITION_DATE_COLUMN = "dt_evento"
DEDUP_COLUMNS = ["bronze_source_file", "bronze_record_hash"]

BRONZE_WRITE_MODE = "overwrite"

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Raw: {SOURCE_PATH}")
print(f"Destino Bronze: {BRONZE_PATH}")
print(f"Modo de escrita: {BRONZE_WRITE_MODE}")

In [0]:
# Lê o CSV da Raw e valida se as colunas esperadas existem.

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

actual_columns = df_source.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: colunas extras encontradas na origem: {extra_columns}")

total_source = df_source.count()

print("Validação inicial OK.")
print(f"Total de registros lidos da Raw: {total_source}")

df_source.printSchema()

In [0]:
# Valida se dt_evento pode gerar as partições ano e mes.

df_validacao_data = (
    df_source
    .withColumn("dt_evento_convertida", to_timestamp(col(PARTITION_DATE_COLUMN)))
    .select(
        count("*").alias("total_linhas"),
        count(when(col(PARTITION_DATE_COLUMN).isNull(), True)).alias("dt_evento_nula"),
        count(
            when(
                col(PARTITION_DATE_COLUMN).isNotNull()
                & col("dt_evento_convertida").isNull(),
                True
            )
        ).alias("falhas_conversao")
    )
)

display(df_validacao_data)

validacao_data = df_validacao_data.collect()[0]

if validacao_data["falhas_conversao"] > 0:
    raise Exception("Existem valores de dt_evento que não foram convertidos para timestamp.")

print("Validação OK: dt_evento pode ser usada para particionamento.")

In [0]:
# Cria a Bronze em memória com dados brutos, auditoria, hash e partições.

hash_columns = [
    coalesce(col(c).cast("string"), lit("__NULL__"))
    for c in EXPECTED_COLUMNS
]

df_bronze = (
    df_source
    .select(
        *[col(c).cast("string").alias(c) for c in EXPECTED_COLUMNS],
        col("_metadata.file_path").alias("bronze_source_file")
    )
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_record_hash", sha2(concat_ws("||", *hash_columns), 256))
    .withColumn("_partition_date", to_timestamp(col(PARTITION_DATE_COLUMN)))
    .withColumn("ano", year(col("_partition_date")))
    .withColumn("mes", month(col("_partition_date")))
    .drop("_partition_date")
)

df_validacao_particao = df_bronze.select(
    count("*").alias("total_linhas"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo")
)

display(df_validacao_particao)

validacao_particao = df_validacao_particao.collect()[0]

if validacao_particao["ano_nulo"] > 0:
    raise Exception("Existem registros com ano nulo na Bronze em memória.")

if validacao_particao["mes_nulo"] > 0:
    raise Exception("Existem registros com mes nulo na Bronze em memória.")

print("DataFrame Bronze criado com sucesso.")

In [0]:
# Remove duplicados do lote para carga full da Bronze.

df_bronze_to_write = df_bronze.dropDuplicates(DEDUP_COLUMNS)

total_source = df_source.count()
total_to_write = df_bronze_to_write.count()

print(f"Total lido da Raw: {total_source}")
print(f"Total após deduplicar o próprio lote: {total_to_write}")
print("Carga full preparada para sobrescrever a Bronze.")

In [0]:
# Grava a Bronze em Delta com overwrite.

(
    df_bronze_to_write
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(BRONZE_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(BRONZE_PATH)
)

print(f"Bronze recriada com sucesso em Delta: {BRONZE_PATH}")
print(f"Modo de escrita utilizado: {BRONZE_WRITE_MODE}")
print(f"Total gravado na Bronze: {total_to_write}")

In [0]:
# Valida volume, partições e campos de auditoria após a escrita full.

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

total_bronze_saved = df_bronze_saved.count()

print(f"Total preparado para gravação: {total_to_write}")
print(f"Total Bronze gravada: {total_bronze_saved}")

if total_bronze_saved != total_to_write:
    raise Exception("Erro: total da Bronze gravada não bate com o total preparado.")

df_validacao_bronze = df_bronze_saved.select(
    count("*").alias("total_linhas"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("bronze_record_hash").isNull(), True)).alias("bronze_record_hash_nulo")
)

display(df_validacao_bronze)

validacao_bronze = df_validacao_bronze.collect()[0]

if validacao_bronze["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Bronze.")

if validacao_bronze["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Bronze.")

if validacao_bronze["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_bronze["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

if validacao_bronze["bronze_record_hash_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_record_hash.")

print("Validação final da Bronze full OK.")